# FAAMG Tech Stocks Core Financial Ratios Analysis
## ACC102 Python Data Product Mini Assignment - Track 4: Interactive Data Analysis Tool

---

**Student Name:** ACC102 Student
**Date:** April 2026
**Topic:** FAAMG Tech Stocks Core Financial Ratios Interactive Analysis Tool
**Target Users:** Year 2 accounting/economics/finance students, beginner investors

---

## Table of Contents
1. [Problem Definition](#1-problem-definition)
2. [Data Loading](#2-data-loading)
3. [Data Cleaning](#3-data-cleaning)
4. [Analysis](#4-analysis)
5. [Visualization](#5-visualization)
6. [Output](#6-output)

---

## 1. Problem Definition

### 1.1 Analytical Problem

This analysis addresses the challenge faced by beginner investors and accounting students in understanding and comparing the financial performance of major technology companies. The FAAMG stocks (Apple, Microsoft, Amazon, Google, Meta) represent some of the most influential companies in the global market, yet their financial metrics can be complex and overwhelming for newcomers.

### 1.2 Target Users

- **Year 2 accounting/economics/finance students** learning about financial statement analysis
- **Beginner investors** seeking to understand fundamental analysis concepts
- **Educators** teaching financial ratio analysis with real-world examples

### 1.3 Research Questions

1. How do the core financial ratios (ROE, Net Profit Margin, Gross Profit Margin, Asset Liability Ratio) vary across FAAMG companies?
2. What trends in profitability and financial leverage can be observed over the 5-year period (2021-2025)?
3. Which company demonstrates the strongest financial performance across different metrics?

### 1.4 Selected Financial Ratios

| Ratio | Formula | Interpretation |
|-------|---------|----------------|
| **ROE (Return on Equity)** | Net Income / Average Shareholders' Equity × 100 | Measures efficiency in generating profits from shareholders' capital |
| **Net Profit Margin** | Net Income / Revenue × 100 | Shows percentage of revenue remaining as profit after all expenses |
| **Gross Profit Margin** | Gross Profit / Revenue × 100 | Indicates production efficiency before operating costs |
| **Asset Liability Ratio** | Total Debt / Total Assets × 100 | Measures financial leverage and capital structure |

In [ ]:
# Import required libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from datetime import datetime

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ All libraries imported successfully!")

## 2. Data Loading

### 2.1 Define Parameters

We will analyze the five FAAMG companies over a 5-year fiscal period (2021-2025).

In [ ]:
# Define target companies (FAAMG)
tickers = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META']

# Company full names mapping
company_names = {
    'AAPL': 'Apple Inc.',
    'MSFT': 'Microsoft Corporation',
    'AMZN': 'Amazon.com Inc.',
    'GOOGL': 'Alphabet Inc.',
    'META': 'Meta Platforms Inc.'
}

# Define analysis period
start_date = '2021-01-01'
end_date = '2025-12-31'
analysis_years = [2021, 2022, 2023, 2024, 2025]

print(f"📅 Analysis Period: {start_date} to {end_date}")
print(f"🏢 Target Companies: {', '.join(company_names.values())}")

### 2.2 Download Financial Data from Yahoo Finance

We use the `yfinance` library to download financial statements including Income Statement, Balance Sheet, and Cash Flow Statement.

In [ ]:
def load_financial_data(ticker, start_date, end_date):
    """
    Load comprehensive financial data for a given ticker.
    
    Args:
        ticker: Stock ticker symbol
        start_date: Start date for historical data
        end_date: End date for historical data
        
    Returns:
        Dictionary containing financial statements
    """
    stock = yf.Ticker(ticker)
    
    data = {
        'info': stock.info,
        'income_stmt': stock.income_stmt,
        'balance_sheet': stock.balance_sheet,
        'cash_flow': stock.cashflow,
        'financials': stock.financials,
        'history': stock.history(start=start_date, end=end_date)
    }
    
    return data

# Load data for all FAAMG companies
print("📥 Loading financial data from Yahoo Finance...")
print("-" * 50)

all_data = {}
for ticker in tickers:
    try:
        data = load_financial_data(ticker, start_date, end_date)
        all_data[ticker] = data
        print(f"✅ {ticker} ({company_names[ticker]}): Data loaded successfully")
    except Exception as e:
        print(f"❌ {ticker}: Failed to load data - {str(e)}")

print("-" * 50)
print(f"📊 Total companies loaded: {len(all_data)}/{len(tickers)}")

### 2.3 Explore Downloaded Data Structure

Let's examine the structure of the financial data for one company as an example.

In [ ]:
# Examine data structure for Apple (AAPL)
print("=" * 60)
print("DATA STRUCTURE EXPLORATION - APPLE INC. (AAPL)")
print("=" * 60)

aapl_data = all_data['AAPL']

# Income Statement
print("\n📄 INCOME STATEMENT (Sample)")
print("-" * 40)
if aapl_data['income_stmt'] is not None and not aapl_data['income_stmt'].empty:
    print(aapl_data['income_stmt'].head(10))
else:
    print("No income statement data available")

# Balance Sheet
print("\n📄 BALANCE SHEET (Sample)")
print("-" * 40)
if aapl_data['balance_sheet'] is not None and not aapl_data['balance_sheet'].empty:
    print(aapl_data['balance_sheet'].head(10))
else:
    print("No balance sheet data available")

## 3. Data Cleaning

### 3.1 Define Financial Ratio Calculation Functions

We implement accurate accounting formulas for each financial ratio.

In [ ]:
def calculate_roe(net_income, shareholders_equity):
    """
    Calculate Return on Equity (ROE).
    
    Formula: ROE = Net Income / Average Shareholders' Equity × 100
    
    Args:
        net_income: Annual net income from income statement
        shareholders_equity: Total shareholders' equity from balance sheet
        
    Returns:
        ROE as percentage (or None if data unavailable)
    """
    if shareholders_equity is None or shareholders_equity == 0:
        return None
    if net_income is None:
        return None
    
    return (abs(net_income) / abs(shareholders_equity)) * 100


def calculate_net_profit_margin(net_income, revenue):
    """
    Calculate Net Profit Margin (NPM).
    
    Formula: NPM = Net Income / Revenue × 100
    
    Args:
        net_income: Annual net income
        revenue: Total revenue
        
    Returns:
        Net Profit Margin as percentage (or None if data unavailable)
    """
    if revenue is None or revenue == 0:
        return None
    if net_income is None:
        return None
    
    return (net_income / revenue) * 100


def calculate_gross_profit_margin(gross_profit, revenue):
    """
    Calculate Gross Profit Margin (GPM).
    
    Formula: GPM = Gross Profit / Revenue × 100
    
    Args:
        gross_profit: Gross profit (Revenue - COGS)
        revenue: Total revenue
        
    Returns:
        Gross Profit Margin as percentage (or None if data unavailable)
    """
    if revenue is None or revenue == 0:
        return None
    if gross_profit is None:
        return None
    
    return (gross_profit / revenue) * 100


def calculate_debt_to_assets(total_debt, total_assets):
    """
    Calculate Asset Liability Ratio (Debt-to-Assets Ratio).
    
    Formula: D/A = Total Debt / Total Assets × 100
    
    Args:
        total_debt: Total debt (short-term + long-term)
        total_assets: Total assets
        
    Returns:
        Asset Liability Ratio as percentage (or None if data unavailable)
    """
    if total_assets is None or total_assets == 0:
        return None
    if total_debt is None:
        return None
    
    return (total_debt / total_assets) * 100

print("✅ Financial ratio calculation functions defined successfully!")

### 3.2 Extract and Calculate Financial Ratios

Now we extract the raw financial figures and calculate the ratios for each company and year.

In [ ]:
def extract_financial_ratios(data_dict, years):
    """
    Extract and calculate all financial ratios from raw financial data.
    
    Args:
        data_dict: Dictionary containing financial data for each ticker
        years: List of years to analyze
        
    Returns:
        DataFrame with all financial ratios
    """
    ratios_data = []
    
    for ticker, company_data in data_dict.items():
        try:
            income_stmt = company_data['income_stmt']
            balance_sheet = company_data['balance_sheet']
            
            if income_stmt is None or income_stmt.empty:
                print(f"⚠️ {ticker}: No income statement data")
                continue
            
            # Get available years from the data
            available_years = income_stmt.columns.tolist()
            
            for year in years:
                year_str = str(year)
                
                # Find the year column
                year_col = None
                for col in available_years:
                    col_str = str(col)
                    if year_str in col_str or col_str.startswith(year_str):
                        year_col = col
                        break
                
                if year_col is None:
                    print(f"⚠️ {ticker}: Year {year} not found in data")
                    continue
                
                # --- Extract Raw Financial Figures ---
                
                # Net Income
                net_income = None
                for idx in income_stmt.index:
                    if 'net income' in str(idx).lower():
                        net_income = income_stmt.loc[idx, year_col]
                        break
                
                # Revenue (search for Total Revenue or Revenue)
                revenue = None
                for idx in income_stmt.index:
                    idx_str = str(idx).lower()
                    if 'revenue' in idx_str or 'total revenue' in idx_str:
                        revenue = income_stmt.loc[idx, year_col]
                        break
                
                # Gross Profit
                gross_profit = None
                for idx in income_stmt.index:
                    if 'gross profit' in str(idx).lower():
                        gross_profit = income_stmt.loc[idx, year_col]
                        break
                
                # Total Assets (from Balance Sheet)
                total_assets = None
                if balance_sheet is not None and not balance_sheet.empty:
                    for idx in balance_sheet.index:
                        if 'total assets' in str(idx).lower():
                            total_assets = balance_sheet.loc[idx, year_col]
                            break
                
                # Total Debt (short-term + long-term)
                total_debt = None
                if balance_sheet is not None and not balance_sheet.empty:
                    short_term_debt = 0
                    long_term_debt = 0
                    
                    for idx in balance_sheet.index:
                        idx_lower = str(idx).lower()
                        if 'short' in idx_lower and 'term' in idx_lower and 'debt' in idx_lower:
                            val = balance_sheet.loc[idx, year_col]
                            short_term_debt = val if pd.notna(val) else 0
                        if 'long' in idx_lower and 'term' in idx_lower and 'debt' in idx_lower:
                            val = balance_sheet.loc[idx, year_col]
                            long_term_debt = val if pd.notna(val) else 0
                        if 'total debt' in idx_lower:
                            total_debt = balance_sheet.loc[idx, year_col]
                            break
                    
                    if total_debt is None:
                        total_debt = short_term_debt + long_term_debt
                
                # Shareholders' Equity
                shareholders_equity = None
                if balance_sheet is not None and not balance_sheet.empty:
                    for idx in balance_sheet.index:
                        if 'total equity' in str(idx).lower() or 'stockholders' in str(idx).lower():
                            shareholders_equity = balance_sheet.loc[idx, year_col]
                            break
                
                # --- Calculate Financial Ratios ---
                roe = calculate_roe(net_income, shareholders_equity)
                net_profit_margin = calculate_net_profit_margin(net_income, revenue)
                gross_profit_margin = calculate_gross_profit_margin(gross_profit, revenue)
                debt_to_assets = calculate_debt_to_assets(total_debt, total_assets)
                
                # Store in results
                ratios_data.append({
                    'Ticker': ticker,
                    'Company': company_names.get(ticker, ticker),
                    'Year': year,
                    'Net Income': net_income,
                    'Revenue': revenue,
                    'Gross Profit': gross_profit,
                    'Total Assets': total_assets,
                    'Total Debt': total_debt,
                    'Shareholders Equity': shareholders_equity,
                    'ROE (%)': round(roe, 2) if roe is not None else None,
                    'Net Profit Margin (%)': round(net_profit_margin, 2) if net_profit_margin is not None else None,
                    'Gross Profit Margin (%)': round(gross_profit_margin, 2) if gross_profit_margin is not None else None,
                    'Asset Liability Ratio (%)': round(debt_to_assets, 2) if debt_to_assets is not None else None
                })
        
        except Exception as e:
            print(f"❌ Error processing {ticker}: {str(e)}")
            continue
    
    return pd.DataFrame(ratios_data)

# Extract and calculate ratios
print("📊 Extracting and calculating financial ratios...")
print("-" * 50)

ratios_df = extract_financial_ratios(all_data, analysis_years)

print(f"\n✅ Successfully processed {len(ratios_df)} records")
print(f"📈 Companies covered: {ratios_df['Ticker'].nunique()}")
print(f"📅 Years covered: {sorted(ratios_df['Year'].unique())}")

### 3.3 Handle Missing Values

Let's check for missing values and handle them appropriately.

In [ ]:
# Check for missing values in ratio columns
ratio_columns = ['ROE (%)', 'Net Profit Margin (%)', 'Gross Profit Margin (%)', 'Asset Liability Ratio (%)']

print("📋 Missing Values Analysis")
print("-" * 50)

for col in ratio_columns:
    missing_count = ratios_df[col].isna().sum()
    missing_pct = (missing_count / len(ratios_df)) * 100
    print(f"{col}: {missing_count} missing ({missing_pct:.1f}%)")

# Display rows with missing values
print("\n📋 Rows with Missing Values:")
missing_rows = ratios_df[ratios_df[ratio_columns].isna().any(axis=1)]
print(missing_rows[['Ticker', 'Year', 'ROE (%)', 'Net Profit Margin (%)', 'Gross Profit Margin (%)', 'Asset Liability Ratio (%)']])

### 3.4 Remove Outliers Using IQR Method

We use the Interquartile Range (IQR) method to identify and handle outliers in our financial ratio data.

In [ ]:
def remove_outliers_iqr(df, columns):
    """
    Remove outliers using the IQR (Interquartile Range) method.
    
    Formula: Outliers are values outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
    
    Args:
        df: Input DataFrame
        columns: List of columns to check for outliers
        
    Returns:
        DataFrame with outliers replaced by None
    """
    df_clean = df.copy()
    
    for col in columns:
        if col in df_clean.columns:
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # Count outliers before replacement
            outliers_count = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
            
            # Replace outliers with NaN
            df_clean.loc[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound), col] = None
            
            if outliers_count > 0:
                print(f"  {col}: {outliers_count} outlier(s) detected and handled")
    
    return df_clean

print("🧹 Removing Outliers Using IQR Method...")
print("-" * 50)

ratios_df_clean = remove_outliers_iqr(ratios_df, ratio_columns)

print("\n✅ Outlier removal complete!")

### 3.5 Final Cleaned Dataset

Let's view the final cleaned dataset.

In [ ]:
# Display the cleaned dataset
print("📊 Final Cleaned Financial Ratios Dataset")
print("=" * 80)

# Sort by Year and Ticker
ratios_df_clean = ratios_df_clean.sort_values(['Year', 'Ticker'])

# Display key columns
display_cols = ['Ticker', 'Company', 'Year', 'ROE (%)', 'Net Profit Margin (%)', 
                 'Gross Profit Margin (%)', 'Asset Liability Ratio (%)']
print(ratios_df_clean[display_cols].to_string(index=False))

## 4. Analysis

### 4.1 Summary Statistics

Let's calculate summary statistics for each financial ratio across all companies.

In [ ]:
# Calculate summary statistics by company
print("📈 Summary Statistics by Company")
print("=" * 80)

summary_by_company = ratios_df_clean.groupby('Ticker')[ratio_columns].agg(['mean', 'std', 'min', 'max'])
print(summary_by_company.round(2))

# Calculate summary statistics by year
print("\n\n📅 Summary Statistics by Year")
print("=" * 80)

summary_by_year = ratios_df_clean.groupby('Year')[ratio_columns].agg(['mean', 'std', 'min', 'max'])
print(summary_by_year.round(2))

### 4.2 Company Comparison Analysis

Let's analyze how each company performs across the different ratios.

In [ ]:
# Calculate average ratios by company (5-year average)
print("📊 5-Year Average Financial Ratios by Company")
print("=" * 80)

avg_by_company = ratios_df_clean.groupby('Ticker')[ratio_columns].mean().round(2)
avg_by_company = avg_by_company.sort_values('ROE (%)', ascending=False)

print(avg_by_company)

# Identify best and worst performers for each ratio
print("\n\n🏆 Best and Worst Performers (by Average)")
print("-" * 50)

for col in ratio_columns:
    best = avg_by_company[col].idxmax()
    best_val = avg_by_company[col].max()
    worst = avg_by_company[col].idxmin()
    worst_val = avg_by_company[col].min()
    
    print(f"\n{col}:")
    print(f"  ✅ Best:  {best} ({best_val:.2f}%)")
    print(f"  ❌ Worst: {worst} ({worst_val:.2f}%)")

### 4.3 Year-over-Year Trend Analysis

Let's analyze how the ratios have changed over the 5-year period.

In [ ]:
# Calculate year-over-year changes
print("📈 Year-over-Year Trend Analysis")
print("=" * 80)

for ticker in tickers:
    print(f"\n{ticker} ({company_names[ticker]})")
    print("-" * 40)
    
    ticker_data = ratios_df_clean[ratios_df_clean['Ticker'] == ticker].sort_values('Year')
    
    if len(ticker_data) >= 2:
        for col in ratio_columns:
            values = ticker_data[col].values
            if len(values) >= 2 and pd.notna(values[0]) and pd.notna(values[-1]):
                change = values[-1] - values[0]
                pct_change = (change / abs(values[0])) * 100 if values[0] != 0 else 0
                trend = "📈" if change > 0 else "📉"
                
                print(f"  {col}: {values[0]:.2f}% → {values[-1]:.2f}% ({trend} {abs(change):.2f}pp, {pct_change:+.1f}%)")

## 5. Visualization

### 5.1 Line Charts - Trend Over Time

Let's create line charts to visualize the trends of each financial ratio over the 5-year period.

In [ ]:
# Set up colors for each company
company_colors = {
    'AAPL': '#555555',    # Apple Gray
    'MSFT': '#00a4ef',    # Microsoft Blue
    'AMZN': '#ff9900',    # Amazon Orange
    'GOOGL': '#4285f4',   # Google Blue
    'META': '#0668e1'     # Meta Blue
}

# Create a figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('FAAMG Financial Ratios Trend Analysis (2021-2025)', fontsize=16, fontweight='bold')

# Plot each ratio
ratio_titles = {
    'ROE (%)': 'Return on Equity (ROE)',
    'Net Profit Margin (%)': 'Net Profit Margin',
    'Gross Profit Margin (%)': 'Gross Profit Margin',
    'Asset Liability Ratio (%)': 'Asset Liability Ratio'
}

for idx, (col, title) in enumerate(ratio_titles.items()):
    ax = axes[idx // 2, idx % 2]
    
    for ticker in tickers:
        ticker_data = ratios_df_clean[ratios_df_clean['Ticker'] == ticker]
        if not ticker_data.empty:
            ax.plot(ticker_data['Year'], ticker_data[col], 
                   marker='o', linewidth=2, markersize=6,
                   color=company_colors[ticker], label=ticker)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Fiscal Year')
    ax.set_ylabel('Percentage (%)')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(analysis_years)

plt.tight_layout()
plt.savefig('faamg_ratios_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Trend visualization saved as 'faamg_ratios_trend.png'")

### 5.2 Bar Charts - Company Comparison

Let's create bar charts for a clear comparison of companies in each year.

In [ ]:
# Create bar charts for each year
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('FAAMG Financial Ratios Comparison by Year', fontsize=16, fontweight='bold')

# Flatten axes for easier iteration
axes_flat = axes.flatten()

# Plot for each year
for idx, year in enumerate(analysis_years):
    ax = axes_flat[idx]
    
    year_data = ratios_df_clean[ratios_df_clean['Year'] == year].set_index('Ticker')
    
    if not year_data.empty:
        x = np.arange(len(year_data))
        width = 0.2
        
        # Plot each ratio as grouped bars
        for i, (col, label) in enumerate([
            ('ROE (%)', 'ROE'),
            ('Net Profit Margin (%)', 'NPM'),
            ('Gross Profit Margin (%)', 'GPM'),
            ('Asset Liability Ratio (%)', 'D/A')
        ]):
            values = year_data[col].values
            ax.bar(x + i * width, values, width, label=label, alpha=0.8)
        
        ax.set_title(f'{year}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Company')
        ax.set_ylabel('Percentage (%)')
        ax.set_xticks(x + width * 1.5)
        ax.set_xticklabels(year_data.index, rotation=45)
        ax.legend(loc='best', fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('faamg_ratios_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Comparison visualization saved as 'faamg_ratios_comparison.png'")

### 5.3 Interactive Charts with Plotly

Let's create interactive visualizations using Plotly for better exploration.

In [ ]:
# Create interactive line chart using Plotly
fig = go.Figure()

for ticker in tickers:
    ticker_data = ratios_df_clean[ratios_df_clean['Ticker'] == ticker].sort_values('Year')
    
    if not ticker_data.empty:
        fig.add_trace(go.Scatter(
            x=ticker_data['Year'],
            y=ticker_data['ROE (%)'],
            mode='lines+markers',
            name=f"{ticker} - ROE",
            line=dict(color=company_colors[ticker], width=2),
            marker=dict(size=8)
        ))

fig.update_layout(
    title='FAAMG Return on Equity (ROE) Trend 2021-2025',
    xaxis_title='Fiscal Year',
    yaxis_title='ROE (%)',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

print("✅ Interactive ROE chart displayed")

In [ ]:
# Create interactive chart for Net Profit Margin
fig = go.Figure()

for ticker in tickers:
    ticker_data = ratios_df_clean[ratios_df_clean['Ticker'] == ticker].sort_values('Year')
    
    if not ticker_data.empty:
        fig.add_trace(go.Scatter(
            x=ticker_data['Year'],
            y=ticker_data['Net Profit Margin (%)'],
            mode='lines+markers',
            name=f"{ticker} - NPM",
            line=dict(color=company_colors[ticker], width=2),
            marker=dict(size=8)
        ))

fig.update_layout(
    title='FAAMG Net Profit Margin Trend 2021-2025',
    xaxis_title='Fiscal Year',
    yaxis_title='Net Profit Margin (%)',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

print("✅ Interactive Net Profit Margin chart displayed")

In [ ]:
# Create grouped bar chart for latest year (2025)
latest_year = 2025
latest_data = ratios_df_clean[ratios_df_clean['Year'] == latest_year]

fig = go.Figure()

ratio_types = [
    ('ROE (%)', 'Return on Equity'),
    ('Net Profit Margin (%)', 'Net Profit Margin'),
    ('Gross Profit Margin (%)', 'Gross Profit Margin'),
    ('Asset Liability Ratio (%)', 'Asset Liability Ratio')
]

for ticker in tickers:
    ticker_data = latest_data[latest_data['Ticker'] == ticker]
    if not ticker_data.empty:
        for col, label in ratio_types:
            value = ticker_data[col].values[0] if col in ticker_data.columns else 0
            if pd.notna(value):
                fig.add_trace(go.Bar(
                    x=[label],
                    y=[value],
                    name=ticker,
                    marker_color=company_colors[ticker]
                ))

fig.update_layout(
    title=f'FAAMG Financial Ratios Comparison - {latest_year}',
    xaxis_title='Financial Ratio',
    yaxis_title='Value (%)',
    barmode='group',
    template='plotly_white',
    height=500
)

fig.show()

print(f"✅ Interactive comparison chart for {latest_year} displayed")

## 6. Output

### 6.1 Export Cleaned Data to CSV

Let's export the cleaned financial ratios data for further use.

In [ ]:
# Export cleaned data to CSV
output_columns = ['Ticker', 'Company', 'Year', 
                  'ROE (%)', 'Net Profit Margin (%)', 
                  'Gross Profit Margin (%)', 'Asset Liability Ratio (%)']

export_df = ratios_df_clean[output_columns].sort_values(['Year', 'Ticker'])
export_df.to_csv('faamg_financial_ratios.csv', index=False)

print("✅ Data exported to 'faamg_financial_ratios.csv'")
print(f"\n📊 Export Summary:")
print(f"   - Total records: {len(export_df)}")
print(f"   - Companies: {export_df['Ticker'].nunique()}")
print(f"   - Years: {sorted(export_df['Year'].unique())}")

# Display exported data
print("\n📋 Exported Data Preview:")
print(export_df.to_string(index=False))

### 6.2 Key Findings Summary

Based on the analysis, here are the key findings:

In [ ]:
# Generate key findings
print("=" * 80)
print("KEY FINDINGS SUMMARY")
print("=" * 80)

# ROE Analysis
roe_avg = avg_by_company['ROE (%)'].sort_values(ascending=False)
print("\n📈 Return on Equity (ROE):")
print(f"   - Highest Average: {roe_avg.index[0]} ({roe_avg.iloc[0]:.2f}%)")
print(f"   - Lowest Average: {roe_avg.index[-1]} ({roe_avg.iloc[-1]:.2f}%)")

# Net Profit Margin Analysis
npm_avg = avg_by_company['Net Profit Margin (%)'].sort_values(ascending=False)
print("\n💰 Net Profit Margin:")
print(f"   - Highest Average: {npm_avg.index[0]} ({npm_avg.iloc[0]:.2f}%)")
print(f"   - Lowest Average: {npm_avg.index[-1]} ({npm_avg.iloc[-1]:.2f}%)")

# Gross Profit Margin Analysis
gpm_avg = avg_by_company['Gross Profit Margin (%)'].sort_values(ascending=False)
print("\n🏭 Gross Profit Margin:")
print(f"   - Highest Average: {gpm_avg.index[0]} ({gpm_avg.iloc[0]:.2f}%)")
print(f"   - Lowest Average: {gpm_avg.index[-1]} ({gpm_avg.iloc[-1]:.2f}%)")

# Asset Liability Ratio Analysis
dta_avg = avg_by_company['Asset Liability Ratio (%)'].sort_values(ascending=False)
print("\n🏦 Asset Liability Ratio (Lower is more conservative):")
print(f"   - Highest (Most Leveraged): {dta_avg.index[0]} ({dta_avg.iloc[0]:.2f}%)")
print(f"   - Lowest (Most Conservative): {dta_avg.index[-1]} ({dta_avg.iloc[-1]:.2f}%)")

### 6.3 Limitations and Future Improvements

**Current Limitations:**
- Data is limited to 5 fiscal years (2021-2025)
- Some financial metrics may have missing values due to differences in reporting
- Outliers were removed using IQR method which may exclude valid extreme values
- Ratios are historical and may not reflect future performance

**Recommended Improvements:**
- Add more companies for broader market comparison
- Include additional financial ratios (P/E, EPS, Current Ratio, etc.)
- Implement interactive dashboard with Streamlit
- Add forecasting capabilities for future trends
- Include industry benchmarks for context

In [ ]:
# Final Summary
print("=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"""
📊 FAAMG Financial Ratios Analysis
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Data Source: Yahoo Finance (yfinance library)
✅ Analysis Period: 2021-2025 (5 fiscal years)
✅ Companies Analyzed: {', '.join(tickers)}
✅ Financial Ratios: ROE, Net Profit Margin, Gross Profit Margin, Asset Liability Ratio
✅ Visualizations: Line charts, Bar charts, Interactive Plotly charts
✅ Output Files: faamg_financial_ratios.csv, faamg_ratios_trend.png, faamg_ratios_comparison.png

📝 Note: This analysis is for educational purposes and should not be considered 
   as financial advice. Always conduct thorough research before making investment decisions.
""")

# Display completion timestamp
from datetime import datetime
print(f"Analysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")